In [1]:
import os
import urllib.request
import zipfile
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import LabelEncoder

In [2]:
url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
zip_path = "ml-100k.zip"

if not os.path.exists(zip_path):
    print("Downloading MovieLens 100k dataset...")
    urllib.request.urlretrieve(url, zip_path)

if not os.path.exists("ml-100k"):
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(".")

In [3]:
print("Loading data...")
r_cols = ['user_id', 'movie_id', 'rating', 'unix_timestamp']
ratings = pd.read_csv('ml-100k/u.data', sep='\t', names=r_cols, encoding='latin-1')

u_cols = ['user_id', 'age', 'gender', 'occupation', 'zip_code']
users = pd.read_csv('ml-100k/u.user', sep='|', names=u_cols, encoding='latin-1')

m_cols = ['movie_id', 'title', 'release_date', 'video_release_date', 'imdb_url', 
          'unknown', 'Action', 'Adventure', 'Animation', 'Childrens', 'Comedy', 
          'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 
          'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
movies = pd.read_csv('ml-100k/u.item', sep='|', names=m_cols, encoding='latin-1')

df = pd.merge(pd.merge(ratings, users, on='user_id'), movies, on='movie_id')

Loading data...


In [4]:
df['liked'] = (df['rating'] >= 4).astype(int)

df['gender_bin'] = (df['gender'] == 'M').astype(int)

le_occ = LabelEncoder()
df['occupation_encoded'] = le_occ.fit_transform(df['occupation'])

features = ['age', 'occupation_encoded', 'gender_bin'] + m_cols[5:]
X = df[features]
y = df['liked']
A = df['gender_bin'] # 1 for M, 0 for F

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
A_test = X_test['gender_bin']

In [6]:
print("Training Random Forest Model...")
clf = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42)
clf.fit(X_train, y_train)

Training Random Forest Model...


RandomForestClassifier(max_depth=10, n_estimators=50, random_state=42)

In [7]:
y_pred = clf.predict(X_test)

In [8]:
print("\n" + "="*40)
print("EVALUATION AND FAIRNESS METRICS")
print("="*40)

cm = confusion_matrix(y_test, y_pred)
print("\n--- 1. Confusion Matrix ---")
print(f"True Negatives: {cm[0][0]}")
print(f"False Positives: {cm[0][1]}")
print(f"False Negatives: {cm[1][0]}")
print(f"True Positives: {cm[1][1]}")

def get_group_metrics(y_true, y_pred, mask):
    y_t = y_true[mask]
    y_p = y_pred[mask]
    
    tp = np.sum((y_t == 1) & (y_p == 1))
    fp = np.sum((y_t == 0) & (y_p == 1))
    tn = np.sum((y_t == 0) & (y_p == 0))
    fn = np.sum((y_t == 1) & (y_p == 0))
    
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    pos_rate = np.mean(y_p) # P(Y_hat=1)
    
    return tpr, fpr, pos_rate

mask_M = (A_test == 1)
mask_F = (A_test == 0)

tpr_M, fpr_M, pr_M = get_group_metrics(y_test, y_pred, mask_M)
tpr_F, fpr_F, pr_F = get_group_metrics(y_test, y_pred, mask_F)

print("\n--- 2. Demographic Parity (Statistical Parity) ---")
print(f"Positive Prediction Rate (Male):   {pr_M:.4f}")
print(f"Positive Prediction Rate (Female): {pr_F:.4f}")
dp_diff = pr_M - pr_F
print(f"Demographic Parity Difference (M - F): {dp_diff:.4f}")

print("\n--- 3. Equalised Odds ---")
print(f"TPR (Male):   {tpr_M:.4f} | FPR (Male):   {fpr_M:.4f}")
print(f"TPR (Female): {tpr_F:.4f} | FPR (Female): {fpr_F:.4f}")
tpr_diff = tpr_M - tpr_F
fpr_diff = fpr_M - fpr_F
print(f"Difference in TPR (M - F): {tpr_diff:.4f}")
print(f"Difference in FPR (M - F): {fpr_diff:.4f}")

print("\n--- 4. Disparate Impact ---")
di = pr_F / pr_M if pr_M > 0 else 0
print(f"Disparate Impact (F / M): {di:.4f}")
if di < 0.8:
    print("Note: Disparate Impact < 0.8 usually indicates potential bias against Females.")
else:
    print("Note: Disparate Impact >= 0.8 indicates acceptable parity by the 80% rule.")

print("\n--- 5. Absolute Average of Odds ---")
aao = 0.5 * (abs(tpr_diff) + abs(fpr_diff))
print(f"Absolute Average of Odds: {aao:.4f}")


EVALUATION AND FAIRNESS METRICS

--- 1. Confusion Matrix ---
True Negatives: 3627
False Positives: 5383
False Negatives: 2414
True Positives: 8576

--- 2. Demographic Parity (Statistical Parity) ---
Positive Prediction Rate (Male):   0.6930
Positive Prediction Rate (Female): 0.7123
Demographic Parity Difference (M - F): -0.0193

--- 3. Equalised Odds ---
TPR (Male):   0.7770 | FPR (Male):   0.5900
TPR (Female): 0.7900 | FPR (Female): 0.6189
Difference in TPR (M - F): -0.0130
Difference in FPR (M - F): -0.0289

--- 4. Disparate Impact ---
Disparate Impact (F / M): 1.0278
Note: Disparate Impact >= 0.8 indicates acceptable parity by the 80% rule.

--- 5. Absolute Average of Odds ---
Absolute Average of Odds: 0.0209
